# residual-skip-add — faded example 3: Build the 1x1 projection shortcut

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `residual-skip-add`. Running the beacon reports progress on the `CNN: Residual skip-connection add` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Residual skip-connection add` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`residual-skip-add`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "residual-skip-add"
DD_SUBTOPIC = "CNN: Residual skip-connection add"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

When the residual branch downsamples (stride 2) and changes channels, the shortcut must be a 1x1 convolution carrying the SAME stride and the new out-channels, so its output shape matches the residual branch and the add is valid.

## Faded exercise 3

The conv branch and forward are written for a shape-changing block. Complete the assignment of `self.skip` as the 1x1 projection conv with the correct out-channels and stride.

**Fill in:** construct the 1x1 Conv2d projection with out_ch channels and the given stride

In [ ]:
import torch.nn as nn
Tensor = t.Tensor


class DownResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.f = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0)

    def forward(self, x: Tensor) -> Tensor:
        return self.f(x) + self.skip(x)


t.manual_seed(0)
block = DownResBlock(16, 32, stride=2)
x = t.randn(2, 16, 32, 32)
out = block(x)

def _test():
    t.manual_seed(0)
    block = DownResBlock(16, 32, stride=2)
    assert isinstance(block.skip, nn.Conv2d), 'skip must be a Conv2d'
    assert block.skip.out_channels == 32, 'skip must lift to out_ch=32'
    assert block.skip.kernel_size == (1, 1) and block.skip.stride == (2, 2), 'must be 1x1 stride-2'
    x = t.randn(2, 16, 32, 32)
    assert block.f(x).shape == block.skip(x).shape, 'branches must agree on shape'
    assert tuple(block(x).shape) == (2, 32, 16, 16)

try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn
Tensor = t.Tensor


class DownResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.f = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0)

    def forward(self, x: Tensor) -> Tensor:
        return self.f(x) + self.skip(x)


t.manual_seed(0)
block = DownResBlock(16, 32, stride=2)
x = t.randn(2, 16, 32, 32)
out = block(x)
```
</details>